# Prior Generating Function
- In the previous notebook, I found the peaks of the LSP signal for three stars.
- These signals matched up with those we know for those stars from the literature.
- There was typically more than one period in each range:
    - short (rot)
    - mid (solar)
    - long (Gleissberg)
- This will formally choose between them.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from astropy.time import Time
from astropy.timeseries import LombScargle

from scipy.signal import find_peaks
from scipy.optimize import curve_fit

from prettytable import PrettyTable

import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

from helpers.LSP_peaks import fit_peaks
from helpers.df_ops import prepare_df, split_df


# Individual stars
raw_HD81809 = pd.read_csv(r"../Data/benchmark/HD81809_Mt_wilson_data.txt", sep='\s',skip_blank_lines=True)
raw_HD160346 = pd.read_csv(r"../Data/benchmark/HD160346_Mt_wilson_data.txt", sep='\s',skip_blank_lines=True)
raw_HD201091 =pd.read_csv(r"../Data/benchmark/HD201091_Mt_wilson_data.txt", sep='\s',skip_blank_lines=True)

data_HD81809 = prepare_df(raw_HD81809)
data_HD160346 = prepare_df(raw_HD160346)
data_HD201091 = prepare_df(raw_HD201091)

train_HD81809,  valid_HD81809,  test_HD81809  = split_df(data_HD81809)
train_HD160346, valid_HD160346, test_HD160346 = split_df(data_HD160346)
train_HD201091, valid_HD201091, test_HD201091 = split_df(data_HD201091)


C:\Users\Joey\AppData\Local\Temp\ipykernel_32636\543762987.py:24: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  raw_HD81809 = pd.read_csv(r"../Data/benchmark/HD81809_Mt_wilson_data.txt", sep='\s',skip_blank_lines=True)
C:\Users\Joey\AppData\Local\Temp\ipykernel_32636\543762987.py:25: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  raw_HD160346 = pd.read_csv(r"../Data/benchmark/HD160346_Mt_wilson_data.txt", sep='\s',skip_blank_lines=True)
C:\Users\Joey\AppData\Local\Temp\ipykernel_32636\543762987.py:26: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separat

In [ ]:
def generate_priors(df,
            manual_freq = 'linear', period_range = [0.1, 100*365], n_periods = 100000,  # this is all the fit_peaks stuff
            FAPs = [10,5,1,0.1], key_FAP_idx = -1, 
            threshold = 5,
            plot_fitpeaks = True, verbose_fitpeaks = True,
            max_peaks = 3,
            verbose_genpriors = True,
            ):
    '''
    Generates the prior guesses for each of the three ranges for the spectrum kernel.

    Takes in the DF, calculates the LSP, identifies the peaks and the number thereof.
    It uses simple heuristics to classify them into long, mid, and short ranges.

    Params
    plot: plots LSP if True
    min_period, max_period in years
    n_periods to set the resolution

    Returns
    The peak periods and their concomitant ranges.
    '''
    peak_periods, peak_heights = fit_peaks(df)

    # Now take the highest peak in each range
    


In [ ]:
def identify_peaks(t=None, y=None, df=None, plot=True, min_period=0.01, max_period=200, n_periods=10000, normalise=True, max_peaks=3):
    '''
    Identifies the peaks available to be used as priors in a GPR fitting.
    Takes in the DF, calculates the LSP, identifies the peaks and the number thereof.
    It uses simple heuristics to classify them into long, mid, and short ranges.

    Params
    plot: plots LSP if True
    min_period, max_period in years
    n_periods to set the resolution

    Returns
    The peak periods and their concomitant ranges.
    '''
    periods = np.linspace(min_period, max_period, n_periods)
    freq = 1 / periods

    if df is not None:
        ls = LombScargle(df['year'], df['sind'])
    elif t is not None and y is not None:
        ls = LombScargle(t, y)
    else:
        raise ValueError("Please define either the df, or t and y")

    powers = ls.power(freq)

    if normalise:
        max_power = max(powers)
        powers /= max_power

    noise = np.percentile(powers, 50)

    peak_idxs, _ = find_peaks(powers, height=0., prominence=noise)
    peak_periods = periods[peak_idxs]
    peak_powers = powers[peak_idxs]

    # Keep only the strongest max_peaks peaks
    if len(peak_periods) > max_peaks:
        top_idx = np.argsort(peak_powers)[-max_peaks:]
        peak_periods = peak_periods[top_idx]
        peak_powers = peak_powers[top_idx]

    # In each range, take the peak with highest power
    peak_info = {}
    peak_data = list(zip(peak_periods, peak_powers))

    peaks_s = [(p, pw) for p, pw in peak_data if p <= 1]
    peaks_m = [(p, pw) for p, pw in peak_data if 1 < p <= 50]
    peaks_l = [(p, pw) for p, pw in peak_data if p > 50]

    if peaks_s:
        peak_info['short'] = max(peaks_s, key=lambda x: x[1])[0]
    if peaks_m:
        peak_info['mid'] = max(peaks_m, key=lambda x: x[1])[0]
    if peaks_l:
        peak_info['long'] = max(peaks_l, key=lambda x: x[1])[0]

    if plot:
        colors = {'short': 'red', 'mid': 'green', 'long': 'orange'}
        fig, ax = plt.subplots(1, figsize=(20, 5))
        ax.scatter(periods, powers, marker='x', color='blue', label="Periodogram")
        for cycle_type, peak_period in peak_info.items():
            ax.axvline(peak_period, color=colors[cycle_type], label=f"Cycle type: {cycle_type}")
        ax.axhline(noise, label="Noise level", color='purple')
        ax.legend()
        plt.show()

    return peak_info